# TxGNN Retrain + Cascade (No Leakage)
Modifies kg_directed.csv BEFORE graph construction to prevent data leakage.

In [1]:
# Cell 1: Install + patches
!pip install dgl==2.4.0+cu121 -f https://data.dgl.ai/wheels/torch-2.1/cu121/repo.html -q
!pip install git+https://github.com/mims-harvard/TxGNN.git -q

import pandas as pd
if not hasattr(pd.DataFrame, 'append'):
    def _append(self, other, ignore_index=False, **kwargs):
        return pd.concat([self, other], ignore_index=ignore_index)
    pd.DataFrame.append = _append
    print('Patched pd.DataFrame.append')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 483.2/483.2 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 104.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 846.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Cell 2: Mount Drive + paths
from google.colab import drive
drive.mount('/content/drive')

SPLIT_DIR = '/content/drive/MyDrive/Colab Notebooks/split'
TXGNN_DIR = '/content/drive/MyDrive/Colab Notebooks/TxGNN_retrained'
DATA_DIR = '/content/drive/MyDrive/Colab Notebooks'

In [ ]:
# Cell 3: Load PrimeKG + splits + matched_pairs
import pandas as pd
import numpy as np
import pickle
from collections import defaultdict

kg = pd.read_csv(f'{DATA_DIR}/kg.csv', low_memory=False)

dp = kg[kg['relation'] == 'disease_phenotype_positive'][['x_index', 'y_index']].drop_duplicates()
disease_phenotypes = defaultdict(set)
for _, row in dp.iterrows():
    disease_phenotypes[int(row['x_index'])].add(int(row['y_index']))

train_ids = np.loadtxt(f'{SPLIT_DIR}/train_disease_ids.txt', dtype=int)
test_ids = np.loadtxt(f'{SPLIT_DIR}/test_disease_ids.txt', dtype=int)
print(f'Train: {len(train_ids)}, Test: {len(test_ids)}')

with open(f'{TXGNN_DIR}/matched_pairs.pkl', 'rb') as f:
    matched_pairs = pickle.load(f)
print(f'Matched pairs: {len(matched_pairs)}')

In [ ]:
# Cell 4: Jaccard similarities
def jaccard(a, b):
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

print('Computing phenotype similarities...')
test_to_train_sims = {}
for i, tid in enumerate(test_ids):
    t_phenos = disease_phenotypes.get(tid, set())
    sims = []
    for tr_id in train_ids:
        tr_phenos = disease_phenotypes.get(tr_id, set())
        s = jaccard(t_phenos, tr_phenos)
        if s > 0:
            sims.append((tr_id, s))
    sims.sort(key=lambda x: x[1], reverse=True)
    test_to_train_sims[tid] = sims
    if (i+1) % 20 == 0:
        top = sims[0][1] if sims else 0
        print(f'  {i+1}/{len(test_ids)}: {len(sims)} matches, top sim={top:.4f}')

print(f'Diseases with match: {sum(1 for v in test_to_train_sims.values() if v)}/{len(test_ids)}')

In [ ]:
# Cell 5: Map PrimeKG IDs to TxGNN IDs
primekg_diseases = kg[kg['x_type']=='disease'][['x_index','x_id']].drop_duplicates()
primekg_diseases_y = kg[kg['y_type']=='disease'][['y_index','y_id']].rename(
    columns={'y_index':'x_index','y_id':'x_id'}).drop_duplicates()
primekg_map = pd.concat([primekg_diseases, primekg_diseases_y]).drop_duplicates('x_index')
idx_to_mondo = dict(zip(primekg_map['x_index'].astype(int), primekg_map['x_id'].astype(str)))

kg_dir = pd.read_csv(f'{DATA_DIR}/kg_directed.csv', low_memory=False)
txd = kg_dir[kg_dir['x_type']=='disease'][['x_idx','x_id']].drop_duplicates()
txd_y = kg_dir[kg_dir['y_type']=='disease'][['y_idx','y_id']].rename(
    columns={'y_idx':'x_idx','y_id':'x_id'}).drop_duplicates()
tx_all = pd.concat([txd, txd_y]).drop_duplicates('x_idx')
tx_mondo_to_idx = {str(row['x_id']): row['x_idx'] for _, row in tx_all.iterrows()}

primekg_to_txgnn = {}
for tr_id in train_ids:
    mondo = idx_to_mondo.get(tr_id)
    if mondo is None:
        continue
    if mondo in tx_mondo_to_idx:
        primekg_to_txgnn[tr_id] = tx_mondo_to_idx[mondo]
        continue
    pm = set(mondo.split('_'))
    for tx_str, tx_idx in tx_mondo_to_idx.items():
        tm = set(str(tx_str).replace('.0','').split('_'))
        if pm & tm:
            primekg_to_txgnn[tr_id] = tx_idx
            break

train_txgnn_to_mondo = {row['x_idx']: str(row['x_id']) for _, row in tx_all.iterrows()}
print(f'Mapped train diseases: {len(primekg_to_txgnn)}/{len(train_ids)}')

In [ ]:
# Cell 6: CRITICAL - Remove test indication edges from kg_directed.csv BEFORE graph construction
import os, shutil

kg_dir_path = f'{DATA_DIR}/kg_directed.csv'
backup_path = f'{DATA_DIR}/kg_directed_backup.csv'

kg_dir = pd.read_csv(kg_dir_path, low_memory=False)
print(f'Original kg_directed.csv: {len(kg_dir)} edges')

test_txgnn_ids = set(r['txgnn_x_idx'] for r in matched_pairs)
mask = (
    ((kg_dir['relation']=='indication') & (kg_dir['y_idx'].isin(test_txgnn_ids))) |
    ((kg_dir['relation']=='rev_indication') & (kg_dir['x_idx'].isin(test_txgnn_ids)))
)
print(f'Test indication edges to remove: {mask.sum()}')

kg_dir_clean = kg_dir[~mask].reset_index(drop=True)
print(f'Modified kg_directed.csv: {len(kg_dir_clean)} edges')

if not os.path.exists(backup_path):
    os.rename(kg_dir_path, backup_path)
    print(f'Backed up original')
else:
    print(f'Backup already exists')
kg_dir_clean.to_csv(kg_dir_path, index=False)

split_path = os.path.join(DATA_DIR, 'full_graph_42')
if os.path.exists(split_path):
    shutil.rmtree(split_path)
    print(f'Deleted cached split')
print('kg_directed.csv modified, ready for clean graph construction')

Original kg_directed.csv: 4050249 edges
Test indication edges to remove: 784
Modified kg_directed.csv: 4049465 edges
Backed up original
Deleted cached split
kg_directed.csv modified, ready for clean graph construction


In [ ]:
# Cell 7: Build TxGNN data from clean CSV + initialize model
from txgnn import TxData, TxGNN, TxEval

TxData_obj = TxData(data_folder_path=DATA_DIR)
TxData_obj.prepare_split(split='full_graph', seed=42)

# Verify no leakage
kg_dir_check = pd.read_csv(f'{DATA_DIR}/kg_directed.csv', low_memory=False)
test_txgnn_ids = set(r['txgnn_x_idx'] for r in matched_pairs)
leaked = kg_dir_check[
    ((kg_dir_check['relation']=='indication') & (kg_dir_check['y_idx'].isin(test_txgnn_ids))) |
    ((kg_dir_check['relation']=='rev_indication') & (kg_dir_check['x_idx'].isin(test_txgnn_ids)))
]
print(f'Test indication edges in graph: {len(leaked)} (MUST be 0)')
assert len(leaked) == 0, 'LEAKAGE DETECTED!'

model = TxGNN(
    data=TxData_obj, weight_bias_track=False,
    proj_name='TxGNN', exp_name='no_leakage', device='cuda:0'
)
model.model_initialize(
    n_hid=512, n_inp=512, n_out=512,
    proto=True, proto_num=3, attention=False,
    sim_measure='all_nodes_profile', bert_measure='disease_name',
    agg_measure='rarity', num_walks=200, walk_mode='bit', path_length=2
)
print('Model initialized on CLEAN graph (no test indication edges)')

Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)


DGL backend not selected or invalid.  Assuming PyTorch for now.


Found local copy...
Found local copy...
Found local copy...
Found saved processed KG... Loading...
Creating splits... it takes several minutes...
split_data_path:  /content/drive/MyDrive/Colab Notebooks/full_graph_42
Creating DGL graph....
Done!
Test indication edges in graph: 0 (MUST be 0)
Model initialized on CLEAN graph (no test indication edges)


In [ ]:
# Cell 8: Patch pretrain for DGL 2.x
import dgl, torch, torch.nn.functional as F, copy, types
from txgnn.TxGNN import Minibatch_NegSampler, get_n_params, get_all_metrics_fb

def pretrain_patched(self, n_epoch=1, learning_rate=1e-3, batch_size=1024,
                     train_print_per_n=20, sweep_wandb=None):
    if self.no_kg:
        raise ValueError('No-KG ablation')
    self.G = self.G.to('cpu')
    print('Creating dataloader...')
    train_eid_dict = {etype: self.G.edges(form='eid', etype=etype) for etype in self.G.canonical_etypes}
    sampler = dgl.dataloading.MultiLayerFullNeighborSampler(2)
    neg_sampler = Minibatch_NegSampler(self.G, 1, 'fix_dst')
    sampler = dgl.dataloading.as_edge_prediction_sampler(sampler, negative_sampler=neg_sampler)
    dataloader = dgl.dataloading.DataLoader(
        self.G, train_eid_dict, sampler,
        batch_size=batch_size, shuffle=True, drop_last=False, num_workers=0
    )
    optimizer = torch.optim.AdamW(self.model.parameters(), lr=learning_rate)
    print(f'Pre-training #param: {get_n_params(self.model)}')
    for epoch in range(n_epoch):
        for step, (nodes, pos_g, neg_g, blocks) in enumerate(dataloader):
            blocks = [i.to(self.device) for i in blocks]
            pos_g = pos_g.to(self.device)
            neg_g = neg_g.to(self.device)
            pred_score_pos, pred_score_neg, pos_score, neg_score = \
                self.model.forward_minibatch(pos_g, neg_g, blocks, self.G, mode='train', pretrain_mode=True)
            scores = torch.cat((pos_score, neg_score)).reshape(-1,)
            labels = [1]*len(pos_score) + [0]*len(neg_score)
            loss = F.binary_cross_entropy(scores, torch.Tensor(labels).float().to(self.device))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if step % train_print_per_n == 0:
                _, _, micro_auroc, _, macro_auroc, _ = get_all_metrics_fb(
                    pred_score_pos, pred_score_neg, scores.reshape(-1,).detach().cpu().numpy(), labels, self.G, True)
                print(f'Epoch {epoch} Step {step} Loss {loss.item():.4f} Micro AUROC {micro_auroc:.4f} Macro AUROC {macro_auroc:.4f}')
    self.best_model = copy.deepcopy(self.model)

model.pretrain = types.MethodType(pretrain_patched, model)
print('Pretrain patched for DGL 2.x')

Pretrain patched for DGL 2.x


In [ ]:
# Cell 9: RETRAIN (2-4 hours)
print('Starting pretrain...')
model.pretrain(n_epoch=1, learning_rate=1e-3, batch_size=1024, train_print_per_n=20)

print('\nStarting finetune...')
model.finetune(n_epoch=200, learning_rate=5e-4, train_print_per_n=5, valid_per_n=20)

import os
save_dir = f'{TXGNN_DIR}/no_leakage_model/'
os.makedirs(save_dir, exist_ok=True)
model.save_model(save_dir)
print('Retraining complete, model saved')

In [ ]:
# Cell 10: Evaluate TxGNN upper bound
evaluator = TxEval(model=model)
test_txgnn_list = [r['txgnn_x_idx'] for r in matched_pairs]
print(f'Evaluating TxGNN on {len(test_txgnn_list)} test diseases...')
test_df = evaluator.eval_disease_centric(
    disease_idxs=test_txgnn_list, relation='indication',
    show_plot=False, verbose=False
)
test_df.index = test_df['ID']
print(f"\nTxGNN (no leakage): AUROC={test_df['AUROC'].mean():.4f}, MRR@10={test_df['MRR@10'].mean():.4f}, R@10={test_df['Recall@10'].mean():.4f}, R@50={test_df['Recall@50'].mean():.4f}")

In [ ]:
# Cell 11: Get train disease scores for cascade
needed = sorted(set(
    primekg_to_txgnn[tr_id]
    for tid in test_ids
    for tr_id, sim in test_to_train_sims.get(tid, [])[:20]
    if tr_id in primekg_to_txgnn
))
print(f'Getting scores for {len(needed)} train diseases...')
train_df = evaluator.eval_disease_centric(
    disease_idxs=needed, relation='indication',
    show_plot=False, verbose=False
)
train_df.index = train_df['ID']
print(f'Done: {len(train_df)} train diseases scored')

In [ ]:
# Cell 12: Cascade evaluation
import ast
from collections import defaultdict

primekg_train_to_mondo = {}
for tr_id, txgnn_idx in primekg_to_txgnn.items():
    mondo = train_txgnn_to_mondo.get(txgnn_idx)
    if mondo:
        primekg_train_to_mondo[tr_id] = mondo
test_primo_to_mondo = {r['test_x_index']: str(r['txgnn_x_id']) for r in matched_pairs}

def cascade_scores(test_primekg_id, top_k):
    sims = test_to_train_sims.get(test_primekg_id, [])
    agg = defaultdict(float)
    w_total = 0.0
    for tr_id, sim in sims[:top_k]:
        mondo = primekg_train_to_mondo.get(tr_id)
        if mondo is None or mondo not in train_df.index:
            continue
        row = train_df.loc[mondo]
        preds = row['Prediction']
        if isinstance(preds, str):
            preds = ast.literal_eval(preds)
        for drug, score in preds.items():
            agg[drug] += sim * score
        w_total += sim
    if w_total > 0:
        return {d: s/w_total for d, s in agg.items()}
    return {}

def eval_ranking(scores_dict, labels_dict, ks=[1,5,10,25,50]):
    drugs = [d for d in scores_dict if d in labels_dict]
    if not drugs:
        return None
    sc = np.array([scores_dict[d] for d in drugs])
    lb = np.array([labels_dict[d] for d in drugs])
    if lb.sum() == 0:
        return None
    order = np.argsort(-sc)
    ranked = lb[order]
    pos = np.where(ranked == 1)[0]
    mrr = 1.0/(pos[0]+1) if len(pos) > 0 else 0.0
    rec = {k: float(ranked[:k].sum()/lb.sum()) for k in ks}
    return mrr, rec

K_VALUES = [1, 3, 5, 10, 20]
k_eval = [1, 5, 10, 25, 50]
print('='*60)
print('CASCADE RESULTS (No Leakage)')
print('='*60)
for K in K_VALUES:
    mrrs = []
    recs = {k: [] for k in k_eval}
    skip = 0
    for mp in matched_pairs:
        pid = mp['test_x_index']
        test_mondo = test_primo_to_mondo.get(pid)
        if test_mondo is None or test_mondo not in test_df.index:
            skip += 1; continue
        cs = cascade_scores(pid, K)
        if not cs:
            skip += 1; continue
        test_row = test_df.loc[test_mondo]
        labels = test_row['Labels']
        if isinstance(labels, str):
            labels = ast.literal_eval(labels)
        if not labels:
            skip += 1; continue
        r = eval_ranking(cs, labels, k_eval)
        if r is None:
            skip += 1; continue
        mrr, rec = r
        mrrs.append(mrr)
        for k in k_eval:
            recs[k].append(rec[k])
    n = len(mrrs)
    print(f'\n--- K={K} (eval {n}, skip {skip}) ---')
    if n > 0:
        print(f'{"Metric":<12} {"Mean":>8}')
        print('-'*22)
        print(f'{"MRR":<12} {np.mean(mrrs):>8.4f}')
        for k in k_eval:
            print(f'{"R@"+str(k):<12} {np.mean(recs[k]):>8.4f}')

In [ ]:
# Cell 13: Save results
import json
results_save = {
    'txgnn': {'auroc': float(test_df['AUROC'].mean()), 'mrr10': float(test_df['MRR@10'].mean()),
              'r10': float(test_df['Recall@10'].mean()), 'r50': float(test_df['Recall@50'].mean())},
    'cascade_k1': {'mrr': float(np.mean(mrrs)), 'r1': float(np.mean(recs[1])),
                   'r10': float(np.mean(recs[10])), 'r50': float(np.mean(recs[50]))}
}
with open(f'{TXGNN_DIR}/no_leakage_results.json', 'w') as f:
    json.dump(results_save, f, indent=2)
print('Results saved')

In [ ]:
# Cell 14: IMPORTANT - Restore original kg_directed.csv
import os
backup_path = f'{DATA_DIR}/kg_directed_backup.csv'
kg_dir_path = f'{DATA_DIR}/kg_directed.csv'
if os.path.exists(backup_path):
    if os.path.exists(kg_dir_path):
        os.remove(kg_dir_path)
    os.rename(backup_path, kg_dir_path)
    print('Restored original kg_directed.csv')

# Also delete the cached split built from modified CSV
split_path = os.path.join(DATA_DIR, 'full_graph_42')
if os.path.exists(split_path):
    shutil.rmtree(split_path)
    print('Deleted cached split (will rebuild from original next time)')
print('Cleanup complete')